# Anomaly Detection to Detect Unusual or Rare Occurrences

Oracle Machine Learning supports anomaly detection to identify rare or unusual records (customers, transactions, etc.). 
In this notebook, we highlight the use of the Expectation Maximization algorithm, which is already supported for clustering. 
The use of EM expands on the One-Class SVM model approach for anomaly detection. EM can capture the underlying data distribution and flag records 
that do not fit the learned data distribution well. An object is identified as an outlier in an EM Anomaly Detection model if its anomaly 
probability is greater than the specified limit, with a default of 0.05. A label of 1 denotes normal, while a label of 0 denotes anomaly. 
We use the customer and demographics data to predict anomalous customers using prob_anomalous.
The entire machine learning methodology runs inside Oracle Autonomous Database (ADB).

Copyright (c) 2025 Oracle Corporation 
###### <a href="https://oss.oracle.com/licenses/upl/" onclick="return ! window.open('https://oss.oracle.com/licenses/upl/');">The Universal Permissive License (UPL), Version 1.0</a>
---

<dl>
<img src="https://www.oracle.com/technetwork/database/options/advanced-analytics/anomalydetection-5663169.jpg" alt="OML Notebooks" width="250"/>
</dl>

* <a href="https://docs.oracle.com/en/cloud/paas/autonomous-data-warehouse-cloud/index.html" target="_blank">Oracle ADB Documentation</a>
* <a href="https://github.com/oracle-samples/oracle-db-examples/tree/main/machine-learning" target="_blank">OML folder on Oracle GitHub</a>
* <a href="https://www.oracle.com/machine-learning" target="_blank">OML Web Page</a>
* <a href="https://www.oracle.com/goto/ml-anomaly-detection" target="_blank">OML Anomaly Detection</a>
* <a href="https://oracle.com/goto/ml-expectation-maximization" target="_blank">OML Expectation Maximization</a>

In [1]:
CREATE OR REPLACE VIEW DEMOGRAPHICS_V AS
  SELECT CUST_ID, YRS_RESIDENCE, EDUCATION, AFFINITY_CARD, 
         HOUSEHOLD_SIZE, OCCUPATION, BOOKKEEPING_APPLICATION, 
         BULK_PACK_DISKETTES, FLAT_PANEL_MONITOR, HOME_THEATER_PACKAGE,
         OS_DOC_SET_KANJI, PRINTER_SUPPLIES, Y_BOX_GAMES
  FROM   SH.SUPPLEMENTARY_DEMOGRAPHICS;

0 row(s) affected


In [2]:
CREATE OR REPLACE VIEW CUSTOMERS360_V AS
  SELECT a.CUST_ID, a.CUST_GENDER, a.CUST_MARITAL_STATUS, a.CUST_YEAR_OF_BIRTH, 
         a.CUST_INCOME_LEVEL, a.CUST_CREDIT_LIMIT, b.EDUCATION, b.AFFINITY_CARD, 
         b.HOUSEHOLD_SIZE, b.OCCUPATION, b.YRS_RESIDENCE, b.Y_BOX_GAMES
  FROM   SH.CUSTOMERS a, DEMOGRAPHICS_V b
  WHERE  a.CUST_ID = b.CUST_ID;

0 row(s) affected


In [3]:
BEGIN DBMS_DATA_MINING.DROP_MODEL('CUSTOMERS360MODEL_AD');
EXCEPTION WHEN OTHERS THEN NULL; END;
/
DECLARE
  v_setlst DBMS_DATA_MINING.SETTING_LIST;
BEGIN
  v_setlst('ALGO_NAME')         := 'ALGO_EXPECTATION_MAXIMIZATION';
  v_setlst('PREP_AUTO')         := 'ON';
  v_setlst('EMCS_OUTLIER_RATE') := '0.1';
        
  DBMS_DATA_MINING.CREATE_MODEL2(
        MODEL_NAME          => 'CUSTOMERS360MODEL_AD',
        MINING_FUNCTION     => 'CLASSIFICATION',
        DATA_QUERY          => 'SELECT * FROM CUSTOMERS360_V',
        CASE_ID_COLUMN_NAME => 'CUST_ID',
        SET_LIST            => v_setlst,
        TARGET_COLUMN_NAME  => NULL); -- NULL target indicates anomaly detection      
END;

0 row(s) affected


## Evaluate the model

In [4]:
SELECT * 
FROM  TABLE(dbms_data_mining.get_model_details_global('CUSTOMERS360MODEL_AD'))
ORDER BY global_detail_name;

GLOBAL_DETAIL_NAME | GLOBAL_DETAIL_VALUE
-------------------+--------------------
LOGLIKELIHOOD      | -3.4040876550830736
NUM_COMPONENTS     | 20                 
RANDOM_SEED        | 0                  
REMOVED_COMPONENTS | 0                  

4 row(s) affected


In [5]:
SELECT * 
FROM (SELECT CUST_ID, round(prob_anomalous,2) prob_anomalous,  
             YRS_RESIDENCE, CUST_MARITAL_STATUS, 
             rank() OVER (ORDER BY prob_anomalous DESC) rnk 
      FROM (SELECT CUST_ID, HOUSEHOLD_SIZE, YRS_RESIDENCE, CUST_GENDER, CUST_MARITAL_STATUS, 
                   prediction_probability(CUSTOMERS360MODEL_AD, '0' USING *) prob_anomalous
            FROM CUSTOMERS360_V))
WHERE rnk <= 5
ORDER BY prob_anomalous DESC;

CUST_ID | PROB_ANOMALOUS | YRS_RESIDENCE | CUST_MARITAL_STATUS | RNK
--------+----------------+---------------+---------------------+----
102926  | 1              | 2             | Married             | 1  
102555  | 1              | 14            | NeverM              | 2  
102287  | 1              | 2             | Mar-AF              | 5  
101478  | 1              | 1             | Married             | 4  
101693  | 1              | 0             | Married             | 3  

5 row(s) affected


In [6]:
CREATE OR REPLACE VIEW EM_ANOMALOUS_RESULTS AS
SELECT * 
FROM (SELECT CUST_ID, anomalous, round(prob_anomalous,2) prob_anomalous, 
             YRS_RESIDENCE, HOUSEHOLD_SIZE, CUST_GENDER,
             CUST_MARITAL_STATUS, 
             RANK() OVER (ORDER BY prob_anomalous DESC) rnk 
      FROM (SELECT CUST_ID, HOUSEHOLD_SIZE, YRS_RESIDENCE, 
                   CUST_GENDER, CUST_MARITAL_STATUS, 
                   prediction(CUSTOMERS360MODEL_AD using *) anomalous,
                   prediction_probability(CUSTOMERS360MODEL_AD, '0' USING *) prob_anomalous
            FROM CUSTOMERS360_V))
ORDER BY prob_anomalous DESC;

0 row(s) affected


In [7]:
SELECT * 
FROM   EM_ANOMALOUS_RESULTS
FETCH FIRST 10 ROWS ONLY;

CUST_ID | ANOMALOUS | PROB_ANOMALOUS | YRS_RESIDENCE | HOUSEHOLD_SIZE | CUST_GENDER | CUST_MARITAL_STATUS | RNK
--------+-----------+----------------+---------------+----------------+-------------+---------------------+----
102926  | 0         | 1              | 2             | 4-5            | F           | Married             | 1  
102555  | 0         | 1              | 14            | 2              | M           | NeverM              | 2  
101693  | 0         | 1              | 0             | 4-5            | F           | Married             | 3  
101478  | 0         | 1              | 1             | 1              | F           | Married             | 4  
102287  | 0         | 1              | 2             | 4-5            | F           | Mar-AF              | 5  
102258  | 0         | 1              | 9             | 2              | F           | NeverM              | 6  
103684  | 0         | 1              | 4             | 2              | M           | Mabsent           

10 row(s) affected


In [8]:
SELECT CUST_ID, PREDICTION,
       RTRIM(TRIM(SUBSTR(OUTPRED."Attribute1",17,100)),'rank="1"/>') FIRST_ATTRIBUTE,
       RTRIM(TRIM(SUBSTR(OUTPRED."Attribute2",17,100)),'rank="2"/>') SECOND_ATTRIBUTE
FROM (SELECT CUST_ID, 
             PREDICTION(CUSTOMERS360MODEL_AD USING *) PREDICTION,
             PREDICTION_DETAILS(CUSTOMERS360MODEL_AD, '0' USING *) PREDICTION_DETAILS 
      FROM   CUSTOMERS360_V
      WHERE  PREDICTION_PROBABILITY(CUSTOMERS360MODEL_AD, '0' USING *) > 0.50
      AND    OCCUPATION = 'TechSup'
      ORDER BY CUST_ID) OUT,
      XMLTABLE('/Details'
                PASSING OUT.PREDICTION_DETAILS
                COLUMNS 
                   "Attribute1" XMLType PATH 'Attribute[1]',
                   "Attribute2" XMLType PATH 'Attribute[2]') OUTPRED
FETCH FIRST 10 ROWS ONLY;

CUST_ID | PREDICTION | FIRST_ATTRIBUTE                                         | SECOND_ATTRIBUTE                               
--------+------------+---------------------------------------------------------+------------------------------------------------
100061  | 0          | "CUST_YEAR_OF_BIRTH" actualValue="1959" weight=".466"   | "Y_BOX_GAMES" actualValue="0" weight="-.036"   
100646  | 0          | "CUST_YEAR_OF_BIRTH" actualValue="1941" weight=".794"   | "Y_BOX_GAMES" actualValue="0" weight="-.053"   
100941  | 0          | "CUST_YEAR_OF_BIRTH" actualValue="1948" weight=".539"   | "EDUCATION" actualValue="9th" weight="-.371"   
101097  | 0          | "Y_BOX_GAMES" actualValue="0" weight="-.001"            | "YRS_RESIDENCE" actualValue="6" weight="-.001" 
101706  | 0          | NULL                                                    | NULL                                           
101970  | 0          | "CUST_YEAR_OF_BIRTH" actualValue="1973" weight=".315"   | "Y_BOX_GAMES" ac

10 row(s) affected


## End of Script